In [11]:
#5. Titan Mining Corp — Lunar Core Quality Prediction
#Context: The Lunar Excavation|Sensor records: 600
#Inputs: Rock_Density (g/cm3); Radiation_Level (mSv); Magnetic_Resonance (0-10)
#Target used for the runnable regression example: Core_Quality_Score — predicted lunar core quality score (0–100).

# ANN Level 1 - FULL PROGRAM (Worksheet + Answers combined)
# 5. Titan Mining Corp — Lunar Core Quality Prediction (Regression)

import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
torch.manual_seed(42)
np.random.seed(42)
print("Libraries imported successfully! PyTorch Version:", torch.__version__)

Libraries imported successfully! PyTorch Version: 2.13.0+cpu


In [12]:
#[CELL 2] Data Pipeline, Normalization & Tensor Conversion

if not os.path.exists('titan_mining_core.csv'):
    n = 600
    demo_df = pd.DataFrame({
        'Rock_Density': np.random.uniform(0.8, 4.0, n),
        'Radiation_Level': np.random.uniform(0, 50, n),
        'Magnetic_Resonance': np.random.uniform(0, 10, n),
    })
    demo_df['Core_Quality_Score'] = np.clip(
        35 + 18*demo_df['Rock_Density'] - 0.8*demo_df['Radiation_Level']
        + 4.5*demo_df['Magnetic_Resonance'] + np.random.normal(0, 5, n), 0, 100
    )
    for col in ['Rock_Density', 'Radiation_Level']:
        mask = np.random.rand(n) < 0.03
        demo_df.loc[mask, col] = np.nan
    demo_df.to_csv('titan_mining_core.csv', index=False)

df = pd.read_csv('titan_mining_core.csv')
feature_cols = ['Rock_Density', 'Radiation_Level', 'Magnetic_Resonance']
X = df[feature_cols].values
y = df['Core_Quality_Score'].values
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)
print(f"X_train shape: {X_train_tensor.shape}, y_train shape: {y_train_tensor.shape}")

X_train shape: torch.Size([480, 3]), y_train shape: torch.Size([480, 1])


In [13]:
# [CELL 3] Designing the ANN Architecture for Regression

class CoreQualityRegressor(nn.Module):
    def __init__(self):
        super(CoreQualityRegressor, self).__init__()
        self.linear1 = nn.Linear(3, 32)
        self.linear2 = nn.Linear(32, 16)
        self.linear3 = nn.Linear(16, 1)
        self.relu = nn.ReLU()
    def forward(self, x):
        x = self.relu(self.linear1(x))
        x = self.relu(self.linear2(x))
        x = self.linear3(x)
        return x
model = CoreQualityRegressor()
print(model)


CoreQualityRegressor(
  (linear1): Linear(in_features=3, out_features=32, bias=True)
  (linear2): Linear(in_features=32, out_features=16, bias=True)
  (linear3): Linear(in_features=16, out_features=1, bias=True)
  (relu): ReLU()
)


In [5]:
# [CELL 4] Loss Function & Optimizer

loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
print("Loss function: MSELoss | Optimizer: Adam")

Loss function: MSELoss | Optimizer: Adam


In [17]:
# [CELL 5] The Training Loop

num_epochs = 100
batch_size = 32
print("Training started...\n")
for epoch in range(num_epochs):
    for i in range(0, len(X_train_tensor), batch_size):
        X_batch = X_train_tensor[i:i+batch_size]
        y_batch = y_train_tensor[i:i+batch_size]
        predictions = model(X_batch)
        loss = loss_fn(predictions, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], MSE Loss: {loss.item():.4f}")
print("\nTraining complete!")

Training started...

Epoch [20/100], MSE Loss: 6455.3042
Epoch [40/100], MSE Loss: 6455.3042
Epoch [60/100], MSE Loss: 6455.3042
Epoch [80/100], MSE Loss: 6455.3042
Epoch [100/100], MSE Loss: 6455.3042

Training complete!


In [18]:
# [CELL 6] The Evaluation Loop

model.eval()
with torch.no_grad():
    test_predictions = model(X_test_tensor)
test_mse = loss_fn(test_predictions, y_test_tensor).item()
test_rmse = np.sqrt(test_mse)
print("Test Set Evaluation:")
print(f" - Mean Squared Error (MSE) : {test_mse:.4f}")
print(f" - Root Mean Squared Error (RMSE): {test_rmse:.4f}\n")
print(f"{'Actual':<15} {'Predicted':<15} {'Abs Error':<15}")
print("-" * 45)
with torch.no_grad():
    for i in range(5):
        actual = y_test_tensor[i].item()
        predicted = test_predictions[i].item()
        error = abs(actual - predicted)
        print(f"{actual:<15.2f} {predicted:<15.2f} {error:<15.2f}")

Test Set Evaluation:
 - Mean Squared Error (MSE) : 6314.9521
 - Root Mean Squared Error (RMSE): 79.4667

Actual          Predicted       Abs Error      
---------------------------------------------
42.77           0.31            42.45          
100.00          0.29            99.71          
66.70           0.28            66.42          
82.22           0.23            81.99          
57.65           0.30            57.35          


In [19]:
# [CELL 7] Real-World Inference

raw_input = np.array([[2.4, 18.0, 6.5]])
imputed_data = imputer.transform(raw_input)
scaled_data = scaler.transform(imputed_data)
input_tensor = torch.tensor(scaled_data, dtype=torch.float32)
model.eval()
with torch.no_grad():
    predicted_score = model(input_tensor).item()
print(f"Predicted Core Quality Score: {predicted_score:.2f} / 100")


Predicted Core Quality Score: 0.28 / 100
